In [1]:
!pip install nano-graphrag

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/917.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.2/285.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.9 MB/s et

In [19]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [1]:
!pip install ollama

In [2]:
!pip install pyngrok

In [4]:
import logging
import socket
import subprocess
import threading
import time
from io import StringIO
from time import time
import numpy as np
import pandas as pd
import torch
import networkx as nx
import ollama
from nano_graphrag import GraphRAG, QueryParam
from nano_graphrag.base import BaseKVStorage
from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs
from sentence_transformers import SentenceTransformer
from flask import Flask, request, jsonify
from pyngrok import ngrok
import nest_asyncio
import re
from google.colab import userdata

ngrok.set_auth_token(userdata.get('ngrok_token'))
logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
logger = logging.getLogger("nano-graphrag")
nest_asyncio.apply()

In [5]:
import time
!pkill -f ollama
!ollama serve > /dev/null 2>&1 &
time.sleep(5)

!ollama pull nomic-embed-text
!ollama pull qwen2

In [7]:
# !ollama show --modelfile qwen2 > Modelfile
# PARAMETER num_ctx 32000
!ollama create -f Modelfile qwen2:ctx32k

In [8]:

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(2)

OLLAMA_API = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2:ctx32k"



In [9]:

async def ollama_model_if_cache(
    prompt: str,
    system_prompt: str = None,
    history_messages=None,
    **kwargs
):
    import json
    import re
    # import ollama


    if history_messages is None:
        history_messages = []

    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    if system_prompt is None:
        system_prompt = ""

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})


    hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
    args_hash = None
    if hashing_kv:
        args_hash = compute_args_hash(MODEL_NAME, messages)
        cached = await hashing_kv.get_by_id(args_hash)
        if cached is not None:
            return cached["return"]

    ollama_client = ollama.AsyncClient()
    try:
        response = await ollama_client.chat(
            model=MODEL_NAME,
            messages=messages,
            **kwargs)
        raw = response.get("message", {}).get("content", "")
        if not raw:
            logger.warning("Ollama returned an empty or malformed message content.")
            raw = '{"nodes": [], "edges": []}'
        print(f"llm output is {raw}")
    except Exception as e:
        logger.error(f"Ollama call failed: {e}")
        raw = '{"nodes": [], "edges": []}'
        print(f"OLLAMA CALL FAILED~~~~~~~~~~~~~~~~~~~~~~~~~ {e}")

    return raw


In [10]:

nest_asyncio.apply()

working_dir = './'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMBED_MODEL = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    cache_folder=working_dir,
    device=device
)

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBED_MODEL.get_sentence_embedding_dimension(),
    max_token_size=EMBED_MODEL.max_seq_length,
)
async def local_embedding(texts: list[str]):
    return EMBED_MODEL.encode(texts, normalize_embeddings=True)



rag = GraphRAG(
    working_dir=working_dir,
    embedding_func=local_embedding,
    enable_llm_cache=True,
    best_model_func=ollama_model_if_cache,
    cheap_model_func=ollama_model_if_cache,
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:nano-graphrag:Load KV full_docs with 12 data
INFO:nano-graphrag:Load KV text_chunks with 39 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 27 data
INFO:nano-graphrag:Loaded graph from ./graph_chunk_entity_relation.graphml with 339 nodes, 194 edges


In [11]:
def extract_from_rag(query, useConcepts=False, graphml_path='graph_chunk_entity_relation.graphml'):

    result = rag.query(
        query,
        param=QueryParam(mode="local", only_need_context=useConcepts)
    )
    if not useConcepts:
      return result

    print(result)

    G = nx.read_graphml(graphml_path)

    entities_section = result.split("-----Entities-----")[1].split("-----Relationships-----")[0].strip()

    if entities_section.startswith("```csv"):
        entities_section = entities_section.replace("```csv", "").replace("```", "").strip()

    entities_df = pd.read_csv(
        StringIO(entities_section),
        sep=',\t',
        quotechar='"',
        skipinitialspace=True,
        on_bad_lines='skip',
        engine='python'
    )

    entities_df.columns = entities_df.columns.str.strip().str.strip('"').str.strip()

    for col in entities_df.columns:
        if entities_df[col].dtype == 'object':
            entities_df[col] = entities_df[col].str.strip().str.strip('"').str.strip()

    relationships_section = result.split("-----Relationships-----")[1].split("-----Sources-----")[0].strip()

    if relationships_section.startswith("```csv"):
        relationships_section = relationships_section.replace("```csv", "").replace("```", "").strip()

    relationships_df = pd.read_csv(
        StringIO(relationships_section),
        sep=',\t',
        quotechar='"',
        skipinitialspace=True,
        on_bad_lines='skip',
        engine='python'
    )

    relationships_df.columns = relationships_df.columns.str.strip().str.strip('"').str.strip()

    for col in relationships_df.columns:
        if relationships_df[col].dtype == 'object':
            relationships_df[col] = relationships_df[col].str.strip().str.strip('"').str.strip()

    entity_type_map = dict(zip(entities_df['entity'], entities_df['type']))
    entity_rank_map = dict(zip(entities_df['entity'], entities_df['rank']))

    relationships_df['source_type'] = relationships_df['source'].map(entity_type_map)
    relationships_df['target_type'] = relationships_df['target'].map(entity_type_map)
    relationships_df['source_rank'] = relationships_df['source'].map(entity_rank_map)
    relationships_df['target_rank'] = relationships_df['target'].map(entity_rank_map)

    def get_relationship_type(source, target):
        source_quoted = f'"{source}"'
        target_quoted = f'"{target}"'

        edge_checks = [
            (source_quoted, target_quoted),
            (target_quoted, source_quoted),
            (source, target),
            (target, source)
        ]

        for src, tgt in edge_checks:
            if G.has_edge(src, tgt):
                edge_data = G[src][tgt]
                for key in ['relationship_type', 'd8', 'type', 'label']:
                    rel_type = edge_data.get(key, None)
                    if rel_type and rel_type != 'UNKNOWN':
                        rel_type_str = str(rel_type).strip().strip('"')
                        if rel_type_str:
                            return rel_type_str

        return 'UNKNOWN'

    relationships_df['relationship_type'] = relationships_df.apply(
        lambda row: get_relationship_type(row['source'], row['target']),
        axis=1
    )

    relationships_df['information_citation'] = "Introduction to AI Course Page"

    return relationships_df

In [12]:

def query_ollama(payload):
    import json
    try:
        response = requests.post(API_URL, json=payload, timeout=120)
        response.raise_for_status()
        result = response.json()

        llm_response = result.get('response', '{}')
        parsed_response = json.loads(llm_response)

        concepts = parsed_response.get('concepts', [])

        return [str(concept) for concept in concepts]

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON response: {str(e)}")
        return []
    except Exception as e:
        print(f"Error querying Ollama: {str(e)}")
        return []

#turns the df into a string to be processed by the llm
def create_enrichment_df_string(df):

    enrichment_df_sorted = df.sort_values('source_rank') if 'source_rank' in df.columns else df

    enriched_context = "=== CONCEPT RELATIONSHIPS (Ordered by Complexity) ===\n\n"
    for idx, row in df.iterrows():
        enriched_context += f"Concept: {row['source']} (Complexity Rank: {row.get('source_rank', 'N/A')})\n"
        enriched_context += f"Relates to: {row['target']} (Complexity Rank: {row.get('target_rank', 'N/A')})\n"
        enriched_context += f"Relationship: {row['relationship_type']}\n"
        enriched_context += f"Description: {row['description']}\n"
        enriched_context += f"Source Citation: {row['information_citation']}\n"
        enriched_context += "-" * 50 + "\n\n"

    return enriched_context


def serialize_dataframe(df):

    if df is None or df.empty:
        return None

    return df.to_dict('records')

def ensure_ollama_running():
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            print("✅ Ollama is running")
            return True
    except:
        pass

    print("Starting Ollama service...")
    subprocess.Popen(['ollama', 'serve'],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL)
    time.sleep(5)
    return True


In [13]:
!pip install flask pyngrok requests


In [14]:
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

In [15]:

import os
os.environ['OLLAMA_NUM_GPU'] = '1'

subprocess.Popen(['ollama', 'serve'],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL)


<Popen: returncode: None args: ['ollama', 'serve']>

In [16]:
!nvidia-smi
!ollama ps

Mon Dec  8 03:59:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             26W /   70W |     206MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

import requests
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        s.listen(1)
        port = s.getsockname()[1]
    return port

port = find_free_port()




app = Flask(__name__)



MODEL_NAME = "qwen2:ctx32k"
API_URL = "http://localhost:11434/api/generate"

system_prompt_extract_concepts = """You are a concept extraction assistant. Extract the key technical concepts, terms, or topics from the user's question.
Return your response as a JSON object with a single key "concepts" containing an array of concept strings.

Example:
User: "How is backprop used in CNNs?"
Response: {"concepts": ["backprop", "CNNs"]}

User: "Explain transformers and attention mechanisms"
Response: {"concepts": ["transformers", "attention mechanisms"]}

Only include the main concepts the user is asking about. Keep concept names concise."""

system_prompt_final_combination = """You are an intelligent educational assistant that provides comprehensive, well-structured answers.

Your task is to answer the user's question by:
1. Starting with foundational/prerequisite concepts first
2. Building up to more complex concepts progressively
3. Integrating information from both the base RAG response and the enriched relationship data
4. Always citing sources using the format [Source: <citation>] after each claim
5. Explaining relationships between concepts when relevant

The enriched data shows concept relationships with:
- source/target: the connected concepts
- relationship_type: how they relate (EXPLAINS, PREREQUISITE_FOR, etc.)
- description: detailed information about the relationship
- information_citation: the source document
- source_rank/target_rank: complexity ranking (lower rank = more foundational)

Structure your answer to flow logically from simpler to more complex concepts, making the learning path clear.
Use concrete examples where helpful. Keep explanations clear and accessible.

Provide a natural, conversational response that directly answers the user's question with citations

USE ONLY RESOURCES FROM THE BASE RESPONSE AND ENRICHMENT. The Source_Citation in the given enrichment holds exactly and EXCLUSIVELY what you should be citing. DO NOT use any more online resources.

"""
!ollama list


!nohup ollama serve > ollama.log 2>&1 &
!sleep 5

!ollama list




@app.route('/query', methods=['POST'])
def query():
    """
    Endpoint to receive user queries and return Ollama responses
    Expected JSON: {"query": "user question here", "context": "optional RAG context"}
    """
    try:
        data = request.get_json()
        user_input = data.get('query', '')
        rag_context = data.get('context', '')

        if not user_input:
            return jsonify({'error': 'No query provided'}), 400

        rag_base_response = extract_from_rag(user_input, useConcepts=False)

        prompt_extract_concepts = f"\n\nUser Question: {user_input}"

        payload_extract_concepts = {
            "model": MODEL_NAME,
            "prompt": prompt_extract_concepts,
            "system": system_prompt_extract_concepts,
            "format": "json",
            "stream": False
        }
        concepts = query_ollama(payload_extract_concepts)

        concepts_string = ", ".join(concepts)

        rag_prompt_extract_concepts = f"Extract all information/documents related to these concepts: {concepts_string}"
        enrichment_df = extract_from_rag(concepts_string, useConcepts=True)
        print(enrichment_df)

        enriched_context = create_enrichment_df_string(enrichment_df)

        final_prompt = f"""User Question: {user_input}

              === BASE CONTEXT ===
              {rag_base_response}

              {enriched_context}

              Please provide a comprehensive answer that:
              1. Starts with the most foundational concepts (lowest rank numbers)
              2. Builds progressively to more complex concepts
              3. Cites sources for all claims using [Source: <citation>] format
              4. Explains how concepts relate to each other
              5. Directly answers the user's question"""

        payload_final_combination = {
            "model": MODEL_NAME,
            "prompt": final_prompt,
            "system": system_prompt_final_combination,
            "stream": False
        }

        print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

        print(final_prompt)

        print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

        try:
            response = requests.post(API_URL, json=payload_final_combination, timeout=120)
            response.raise_for_status()
            result = response.json()

            llm_response = result.get('response', '')

            return jsonify({
                'response': llm_response,
                'enrichment_df': serialize_dataframe(enrichment_df),
                'status': 'success'
            })

        except Exception as e:
            print(f"Error getting final response: {str(e)}")
            return jsonify({
                'error': 'Failed to generate response',
                'status': 'error'
            }), 500

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({
            'error': str(e),
            'status': 'error'
        }), 500



public_url = ngrok.connect(port)
print(f"PUBLIC URL: {public_url}")


def run_flask():
    app.run(port=port, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("Ngrok tunnel is active")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Server stopped")